In [1]:
!pip install paho-mqtt requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.2/67.2 kB 2.8 MB/s eta 0:00:00


In [17]:
import requests

AI_API = "https://ai-smart-retriage-api.onrender.com/predict"

payload = {
    "age": 55,
    "initial_triage_level": 3,
    "systolic_bp": 80,
    "diastolic_bp": 50,
    "pulse": 100,
    "rr": 22,
    "spo2": 94,
    "temp": 23.0,
    "wait_minutes": 25,
    "repeated_measure_count": 2,
    "delta_pulse": -1,
    "delta_bp": 0
}

r = requests.post(AI_API, json=payload, timeout=120)
print(r.status_code)
print(r.text)

200
{"recommendation":"retriage_now","status_color":"red","probabilities":{"continue_waiting":0.16,"observe_closely":0.015,"retriage_now":0.825}}


In [ ]:
import ssl
import json
import requests
from datetime import datetime
import paho.mqtt.client as mqtt

# ===== HiveMQ Cloud =====
BROKER = "aa6612317d6a41b4ac5da80f9588279e.s1.eu.hivemq.cloud"
PORT = 8883
USERNAME = "rxsamart"
PASSWORD = "MedTU117"   # ต้องตรงกับ HiveMQ เป๊ะ
TOPIC = "er/sensehat/PT-001"

# ===== AI API =====
AI_API = "https://ai-smart-retriage-api.onrender.com/predict"

# ===== ThingsBoard =====
TB_HOST = "https://thingsboard.cloud"
TB_TOKEN = "m0Wsavfu4EkqorxaZT0v"   # PT-001 token

last_state = {
    "pulse": None,
    "systolic_bp": None
}

def send_to_thingsboard(payload):
    url = f"{TB_HOST}/api/v1/{TB_TOKEN}/telemetry"
    r = requests.post(url, json=payload, timeout=30)
    print("TB status:", r.status_code)
    if r.text:
        print("TB response:", r.text)

def map_sensor_to_model_input(sensor_payload):
    temperature = sensor_payload["temperature"]
    pressure = sensor_payload["pressure"]
    humidity = sensor_payload["humidity"]

    # mapping demo
    temp = round(temperature, 1)

    # pressure -> systolic_bp (สมมุติ)
    systolic_bp = int((pressure - 900) * 0.8)
    systolic_bp = max(80, min(systolic_bp, 140))

    # humidity -> pulse (สมมุติ)
    pulse = int(60 + (humidity * 0.8))
    pulse = max(60, min(pulse, 140))

    prev_pulse = last_state["pulse"] if last_state["pulse"] is not None else pulse
    prev_bp = last_state["systolic_bp"] if last_state["systolic_bp"] is not None else systolic_bp

    delta_pulse = pulse - prev_pulse
    delta_bp = systolic_bp - prev_bp

    model_input = {
        "age": 55,
        "initial_triage_level": 3,
        "systolic_bp": systolic_bp,
        "diastolic_bp": max(50, systolic_bp - 40),
        "pulse": pulse,
        "rr": 18 if pulse < 100 else 22,
        "spo2": 98 if pulse < 95 else (94 if pulse < 115 else 90),
        "temp": temp,
        "wait_minutes": 15 if sensor_payload["status_color"] == "green" else (40 if sensor_payload["status_color"] == "yellow" else 25),
        "repeated_measure_count": 1 if sensor_payload["status_color"] == "green" else 2,
        "delta_pulse": delta_pulse,
        "delta_bp": delta_bp
    }

    last_state["pulse"] = pulse
    last_state["systolic_bp"] = systolic_bp

    return model_input

def on_connect(client, userdata, flags, rc):
    print("Connected with result code:", rc)
    if rc == 0:
        client.subscribe(TOPIC)
        print("Subscribed to:", TOPIC)
    else:
        print("MQTT connect failed")

def on_message(client, userdata, msg):
    try:
        sensor_payload = json.loads(msg.payload.decode())

        print("Topic:", msg.topic)
        print("Raw payload:", sensor_payload)

        model_input = map_sensor_to_model_input(sensor_payload)
        print("Model input:", model_input)

        ai_response = requests.post(AI_API, json=model_input, timeout=120)
        print("AI status:", ai_response.status_code)
        ai_response.raise_for_status()
        ai_result = ai_response.json()
        print("AI result:", ai_result)

        dashboard_payload = {
            "patient_name": "Patient Green",
            "patient_id": sensor_payload["patient_id"],
            "temperature": sensor_payload["temperature"],
            "pressure": sensor_payload["pressure"],
            "humidity": sensor_payload["humidity"],
            "status_from_device": sensor_payload["status_color"],

            "age": model_input["age"],
            "systolic_bp": model_input["systolic_bp"],
            "diastolic_bp": model_input["diastolic_bp"],
            "pulse": model_input["pulse"],
            "spo2": model_input["spo2"],
            "wait_minutes": model_input["wait_minutes"],

            "ai_recommendation": ai_result["recommendation"],
            "status_color": ai_result["status_color"],
            "timestamp_text": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        }

        send_to_thingsboard(dashboard_payload)
        print("Processed for dashboard:", dashboard_payload)
        print("-" * 60)

    except Exception as e:
        print("Error:", e)

client = mqtt.Client()
client.username_pw_set(USERNAME, PASSWORD)
client.tls_set(tls_version=ssl.PROTOCOL_TLS_CLIENT)
client.on_connect = on_connect
client.on_message = on_message

client.connect(BROKER, PORT, 60)
client.loop_forever()

Topic: er/sensehat/PT-001
Raw payload: {'patient_id': 'PT-001', 'device_type': 'sensehat_emulator', 'temperature': 23.2, 'pressure': 663.4, 'humidity': 38.2, 'status_color': 'red', 'timestamp_text': '2026-04-26 21:17:21'}
Model input: {'age': 55, 'initial_triage_level': 3, 'systolic_bp': 80, 'diastolic_bp': 50, 'pulse': 90, 'rr': 18, 'spo2': 98, 'temp': 23.2, 'wait_minutes': 25, 'repeated_measure_count': 2, 'delta_pulse': 0, 'delta_bp': 0}
AI status: 200
AI result: {'recommendation': 'retriage_now', 'status_color': 'red', 'probabilities': {'continue_waiting': 0.16, 'observe_closely': 0.01, 'retriage_now': 0.83}}
TB status: 200
Processed for dashboard: {'patient_name': 'Patient Green', 'patient_id': 'PT-001', 'temperature': 23.2, 'pressure': 663.4, 'humidity': 38.2, 'status_from_device': 'red', 'age': 55, 'systolic_bp': 80, 'diastolic_bp': 50, 'pulse': 90, 'spo2': 98, 'wait_minutes': 25, 'ai_recommendation': 'retriage_now', 'status_color': 'red', 'timestamp_text': '2026-04-26 14:17:22'}